# Case 1 — Repeat Alarm Forecast (Snowflake Notebook)

**Amar Bank — Workshop Part 2**

Notebook ini mengonversi analisa forecast *repeat* (jumlah harian nasabah/aplikasi yang mengulang) dari notebook Python lokal menjadi **Snowflake Notebook** yang berjalan langsung di atas data Snowflake.

**Alur:**
1. Baca data harian dari `AMAR_WORKSHOP_P2.REPEAT.REPEAT_DAILY_COUNT`
2. Eksplorasi deret waktu (EDA)
3. Baseline **Linear Regression**
4. Uji stasioneritas (**ADF**), **ACF/PACF**, dekomposisi musiman
5. Model **SARIMAX** (statsmodels) — *tanpa pmdarima*, order manual
6. Evaluasi (MAE / RMSE) & forecast ke depan
7. Simpan hasil ke tabel `REPEAT_FORECAST`

> `pmdarima.auto_arima` **tidak tersedia** di Anaconda Snowflake, jadi order SARIMAX ditetapkan manual (divalidasi lewat ACF/PACF & ADF).

**Runtime:** notebook ini berjalan di **Container Runtime / SPCS** (bukan warehouse runtime), jadi **tidak ada tombol Packages**. Paket `pandas`, `numpy`, `matplotlib`, `scikit-learn` sudah **pra-instal**; `plotly` dan `statsmodels` di-`pip install` pada cell pertama di bawah. Bila Container Runtime Anda *air-gapped* (tak bisa akses `pypi.org`), pra-instal paket tsb ke image/compute pool.

In [ ]:
# Container Runtime / SPCS: install paket yang belum pra-instal (BUKAN via tombol Packages).
# pandas/numpy/matplotlib/scikit-learn sudah tersedia di image; plotly & statsmodels belum.
!pip install plotly statsmodels

In [ ]:
# Import library & ambil Snowpark session aktif
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Container Runtime / SPCS: session belum tentu di DB/schema workshop -> set eksplisit.
session.use_database('AMAR_WORKSHOP_P2')
session.use_schema('REPEAT')
print('Session OK ->', session.get_current_database(), session.get_current_schema())

## 1. Baca data dari Snowflake
Ambil tabel agregat harian `REPEAT_DAILY_COUNT` (1 baris = 1 hari) ke pandas.

In [ ]:
df = session.table('AMAR_WORKSHOP_P2.REPEAT.REPEAT_DAILY_COUNT').to_pandas()
df.columns = [c.lower() for c in df.columns]
df['created_date'] = pd.to_datetime(df['created_date'])
df = df.sort_values('created_date').reset_index(drop=True)
print('Rows:', len(df), '| Range:', df.created_date.min().date(), '->', df.created_date.max().date())
df.head()

## 2. Eksplorasi deret waktu (EDA)
Plot volume repeat harian: perhatikan **trend** dan pola **musiman** (mingguan/tahunan).

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.created_date, y=df.repeat_count, mode='lines', name='repeat_count'))
fig.update_layout(title='Repeat Count Harian', xaxis_title='Tanggal', yaxis_title='Repeat Count', height=400)
fig.show()

## 3. Train / Test split
Latih sampai **akhir 2025**, uji pada **2026** (out-of-sample). Index tanggal di-set `freq='D'` agar musiman dikenali.

In [ ]:
ts = df.set_index('created_date')['repeat_count'].asfreq('D')
ts = ts.interpolate()  # jaga bila ada tanggal bolong
SPLIT_DATE = '2026-01-01'
train = ts[ts.index < SPLIT_DATE]
test  = ts[ts.index >= SPLIT_DATE]
print('Train:', len(train), '| Test:', len(test))

## 4. Baseline — Linear Regression
Regresi linear terhadap **ordinal tanggal** (tren lurus). Menangkap trend tapi tidak musiman — pembanding untuk SARIMAX.

In [ ]:
Xtr = train.index.map(pd.Timestamp.toordinal).to_numpy().reshape(-1, 1)
Xte = test.index.map(pd.Timestamp.toordinal).to_numpy().reshape(-1, 1)
lin = LinearRegression().fit(Xtr, train.values)
pred_lin = lin.predict(Xte)
print('Linear  MAE :', round(mean_absolute_error(test.values, pred_lin), 2))
print('Linear  RMSE:', round(mean_squared_error(test.values, pred_lin) ** 0.5, 2))
print('Linear  R2  :', round(r2_score(test.values, pred_lin), 3))

## 5. Uji stasioneritas — ADF
**p-value < 0.05** ⇒ deret stasioner. Bila belum, naikkan orde diferensiasi `d`.

In [ ]:
res = adfuller(train, autolag='AIC')
print('ADF Statistic :', round(res[0], 4))
print('p-value       :', round(res[1], 6))
print('# lags        :', res[2])
for k, v in res[4].items():
    print(f'  crit {k}: {round(v,3)}')
print('=> STASIONER' if res[1] < 0.05 else '=> BELUM stasioner (pertimbangkan d=1)')

## 6. ACF & PACF
Pilih orde **MA (q)** dari ACF dan **AR (p)** dari PACF. Puncak tiap ~7 lag = musiman mingguan (m=7).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(train, lags=40, ax=ax[0]); ax[0].set_title('ACF')
plot_pacf(train, lags=40, ax=ax[1]); ax[1].set_title('PACF')
plt.tight_layout(); plt.show()

## 7. Dekomposisi musiman
Pecah deret jadi **trend + seasonal + residual**, `period=7` (mingguan).

In [ ]:
dec = seasonal_decompose(train, model='additive', period=7)
dec.plot(); plt.tight_layout(); plt.show()

## 8. Model SARIMAX (statsmodels)
Order **manual** (pengganti auto_arima):
- Non-musiman `(p,d,q) = (2,0,2)`
- Musiman mingguan `(P,D,Q,m) = (1,1,1,7)`

> **Teaching point:** untuk data **harian**, musiman yang tepat = **m=7** (mingguan), bukan m=12 (bulanan).

In [ ]:
ORDER = (2, 0, 2)
SEASONAL = (1, 1, 1, 7)
model = SARIMAX(train, order=ORDER, seasonal_order=SEASONAL,
                enforce_stationarity=False, enforce_invertibility=False)
result = model.fit(disp=False)
print(result.summary())

## 9. Evaluasi out-of-sample (2026)
Bandingkan prediksi vs actual, hitung **MAE & RMSE**, lalu plot dengan interval kepercayaan.

In [ ]:
fc = result.get_forecast(steps=len(test))
pred = fc.predicted_mean
ci = fc.conf_int()
mae = mean_absolute_error(test.values, pred.values)
rmse = mean_squared_error(test.values, pred.values) ** 0.5
print('SARIMAX MAE :', round(mae, 2))
print('SARIMAX RMSE:', round(rmse, 2))

fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index[-120:], y=train.values[-120:], name='Train (ekor)'))
fig.add_trace(go.Scatter(x=test.index, y=test.values, name='Actual (test)'))
fig.add_trace(go.Scatter(x=pred.index, y=pred.values, name='SARIMAX Predicted'))
fig.add_trace(go.Scatter(x=ci.index, y=ci.iloc[:,0], line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(x=ci.index, y=ci.iloc[:,1], fill='tonexty', line=dict(width=0),
                         name='95% CI', fillcolor='rgba(0,150,200,0.2)'))
fig.update_layout(title='SARIMAX: Actual vs Predicted', height=420)
fig.show()

## 10. Forecast ke depan (30 hari)
Latih ulang di **seluruh data** lalu forecast 30 hari — output untuk *repeat alarm*.

In [ ]:
H = 30
final_model = SARIMAX(ts, order=ORDER, seasonal_order=SEASONAL,
                      enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
ffc = final_model.get_forecast(steps=H)
future = ffc.predicted_mean
fci = ffc.conf_int()

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts.index[-150:], y=ts.values[-150:], name='History'))
fig.add_trace(go.Scatter(x=future.index, y=future.values, name='Forecast 30d'))
fig.add_trace(go.Scatter(x=fci.index, y=fci.iloc[:,0], line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(x=fci.index, y=fci.iloc[:,1], fill='tonexty', line=dict(width=0),
                         name='95% CI', fillcolor='rgba(0,180,120,0.2)'))
fig.update_layout(title='Forecast Repeat 30 Hari ke Depan', height=420)
fig.show()

## 11. Simpan hasil forecast ke Snowflake
Tulis prediksi (test + future) ke tabel `REPEAT_FORECAST` untuk dashboard / alerting / Task.

In [ ]:
out = pd.DataFrame({
    'FORECAST_DATE': list(pred.index) + list(future.index),
    'FORECAST_VALUE': list(pred.values) + list(future.values),
    'LOWER_CI': list(ci.iloc[:,0].values) + list(fci.iloc[:,0].values),
    'UPPER_CI': list(ci.iloc[:,1].values) + list(fci.iloc[:,1].values),
    'SEGMENT': ['TEST']*len(pred) + ['FUTURE']*len(future),
})
out['FORECAST_DATE'] = pd.to_datetime(out['FORECAST_DATE']).dt.date
out['FORECAST_VALUE'] = out['FORECAST_VALUE'].round(0)
session.write_pandas(out, 'REPEAT_FORECAST', auto_create_table=True, overwrite=True,
                     database='AMAR_WORKSHOP_P2', schema='REPEAT')
print('Tersimpan ke REPEAT_FORECAST:', len(out), 'baris')
session.table('AMAR_WORKSHOP_P2.REPEAT.REPEAT_FORECAST').limit(5).show()

## 12. Deploy model ke Model Registry

SARIMAX (statsmodels) bukan estimator bawaan, jadi dibungkus **`CustomModel`** lalu didaftarkan ke **Model Registry**. Model final (fit di seluruh data) di-pickle sebagai artifact; method `predict` menerima kolom `STEP` (1..H) dan mengembalikan forecast + CI per langkah (jumlah baris output = input, agar cocok untuk batch inference).

In [ ]:
# --- Simpan model SARIMAX ke Model Registry via CustomModel ---
import os, joblib
from snowflake.ml.model import custom_model
from snowflake.ml.registry import Registry

MODEL_DIR = '/tmp/sarimax_repeat'
os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(final_model, os.path.join(MODEL_DIR, 'sarimax.joblib'))   # final_model dari cell forecast (fit seluruh data)

class SarimaxForecaster(custom_model.CustomModel):
    def __init__(self, context):
        super().__init__(context)
        self._model = joblib.load(context.path('sarimax'))

    @custom_model.inference_api
    def predict(self, X: pd.DataFrame) -> pd.DataFrame:
        steps = int(X['STEP'].max())
        fc = self._model.get_forecast(steps=steps)
        mean = fc.predicted_mean.reset_index(drop=True)
        ci = fc.conf_int().reset_index(drop=True)
        pos = X['STEP'].astype(int).to_numpy() - 1
        return pd.DataFrame({
            'FORECAST_VALUE': mean.to_numpy()[pos].round(0),
            'LOWER_CI': ci.iloc[:, 0].to_numpy()[pos],
            'UPPER_CI': ci.iloc[:, 1].to_numpy()[pos],
        })

ctx = custom_model.ModelContext(artifacts={'sarimax': os.path.join(MODEL_DIR, 'sarimax.joblib')})
sarimax_fc = SarimaxForecaster(ctx)

sample = pd.DataFrame({'STEP': list(range(1, H + 1))})
print('Sanity check (lokal):')
print(sarimax_fc.predict(sample).head())

reg = Registry(session, database_name='AMAR_WORKSHOP_P2', schema_name='REPEAT')
try:
    session.sql('DROP MODEL IF EXISTS AMAR_WORKSHOP_P2.REPEAT.REPEAT_SARIMAX_FORECAST').collect()
except Exception as e:
    print('drop skipped:', e)

mv = reg.log_model(
    sarimax_fc,
    model_name='REPEAT_SARIMAX_FORECAST', version_name='V1',
    sample_input_data=sample,
    conda_dependencies=['statsmodels', 'pandas', 'numpy'],
    target_platforms=['WAREHOUSE'],
    comment='SARIMAX (2,0,2)(1,1,1,7) forecast repeat harian',
    metrics={'mae': float(mae), 'rmse': float(rmse)},
)
reg.get_model('REPEAT_SARIMAX_FORECAST').default = 'V1'
print('Registered REPEAT_SARIMAX_FORECAST V1  ->  metrics:', {'mae': round(float(mae),2), 'rmse': round(float(rmse),2)})

## 13. Uji batch inference dari Model Registry

Panggil model terdaftar via `mv.run(..., function_name='predict')` — dieksekusi **di warehouse**. Input = daftar `STEP` (1..H), output = forecast + interval kepercayaan. Hasil disimpan ke tabel untuk dashboard/alerting.

In [ ]:
# --- Batch inference dari Model Registry (jalan di warehouse) ---
mv = reg.get_model('REPEAT_SARIMAX_FORECAST').version('V1')

future_steps = session.create_dataframe(pd.DataFrame({'STEP': list(range(1, H + 1))}))
scored = mv.run(future_steps, function_name='predict')
scored.show()

scored.write.mode('overwrite').save_as_table('AMAR_WORKSHOP_P2.REPEAT.REPEAT_FORECAST_FROM_REGISTRY')
print('Batch inference OK -> AMAR_WORKSHOP_P2.REPEAT.REPEAT_FORECAST_FROM_REGISTRY')

## 14. (Bonus) Forecast per segmen & langkah berikutnya
`REPEAT_DAILY_COUNT_SEGMENT` (product × city × repeat_status) bisa diforecast per grup (loop SARIMAX) untuk *alarm* granular.

**Selanjutnya:** jadwalkan notebook ini (Snowflake Notebook schedule / Task) agar `REPEAT_FORECAST` selalu update, lalu bandingkan actual harian vs `LOWER_CI`/`UPPER_CI` untuk memicu *alarm*.

✅ Selesai — Case 1.